In [116]:

"""
================================================================================
LIQUID HANDLER TRANSFER FILE GENERATOR
================================================================================

PURPOSE:
--------
This script generates transfer instructions for automated liquid handlers.
It calculates how much volume of each stock solution needs to be transferred
to each destination well to achieve target concentrations.

VISUAL OVERVIEW:
----------------
Imagine you're making cocktails, but with chemicals instead of alcohol:

SOURCE PLATES (Stock Solutions):
  Plate s1 (High concentration):     Plate s4 (Low concentration):
    A1: Glucose 100 mM                  A1: Glucose 10 mM
    B2: Salt 50 mM                     B2: Salt 5 mM
    C3: Phosphate 100 mM              C3: Phosphate 10 mM

DESTINATION PLATES (Final Solutions):
  Plate dest_1:
    A1: Need 5 mM Glucose, 2 mM Salt, 1 mM Phosphate
    A2: Need 10 mM Glucose, 1 mM Salt, 0.5 mM Phosphate
    ... (many more wells)

WHAT THIS SCRIPT DOES:
  For each destination well, it calculates:
    - How much to transfer from each stock (e.g., 75 µL from s1:A1)
    - Which stock to use (high or low concentration)
    - How much water to add (to reach 1500 µL total)
    - Creates a list of transfer commands for the liquid handler

WHAT IT DOES (DETAILED):
------------------------
1. Reads input files:
   - Stock concentrations (what concentrations are available)
   - Stock plate layouts (where stocks are located on plates)
   - Target concentrations (what we want in each destination well)

2. Calculates transfer volumes:
   - For each destination well and each component, calculates how much volume
     to transfer from stock solutions
   - Chooses between high/low concentration stocks based on minimum volume requirements
   - Adds water to fill each well to the target total volume

3. Generates transfer instructions:
   - Creates a CSV file with transfer commands for the liquid handler
   - Each row = one transfer: source plate/well → destination plate/well + volume
   - Splits large transfers that exceed maximum tip volume

4. Validates results:
   - Checks that volumes sum correctly
   - Verifies concentrations match targets
   - Warns about source well depletion

EXAMPLE OUTPUT (transfer_instructions.csv):
--------------------------------------------
Source_Plate, Source_Well, Dest_Plate, Dest_Well, Transfer_Vol
s1,          A1,          dest_1,      A1,        75.0
s1,          B2,          dest_1,      A1,        60.0
s1,          C3,          dest_1,      A1,        15.0
s1,          A1,          dest_1,      A1,        15.0  (culture)
s_water,     A1,          dest_1,      A1,        1335.0
...
This tells the liquid handler: "Transfer 75 µL from s1:A1 to dest_1:A1"

INPUT FILES:
------------
- stock_concentrations.csv: List of all components and their stock concentrations
- 24-well_stock_plate_high.csv: Layout of high-concentration stock plate (plate s1)
- 24-well_stock_plate_low.csv: Layout of low-concentration stock plate (plate s4)
- target_concentrations.csv: Target concentration for each component in each well

OUTPUT FILE:
------------
- transfer_instructions.csv: Transfer commands for liquid handler
  Columns: Source_Plate, Source_Well, Dest_Plate, Dest_Well, Transfer_Vol

================================================================================
"""

'\n================================================================================\nLIQUID HANDLER TRANSFER FILE GENERATOR\n================================================================================\n\nPURPOSE:\n--------\nThis script generates transfer instructions for automated liquid handlers.\nIt calculates how much volume of each stock solution needs to be transferred\nto each destination well to achieve target concentrations.\n\nVISUAL OVERVIEW:\n----------------\nImagine you\'re making cocktails, but with chemicals instead of alcohol:\n\nSOURCE PLATES (Stock Solutions):\n  Plate s1 (High concentration):     Plate s4 (Low concentration):\n    A1: Glucose 100 mM                  A1: Glucose 10 mM\n    B2: Salt 50 mM                     B2: Salt 5 mM\n    C3: Phosphate 100 mM              C3: Phosphate 10 mM\n\nDESTINATION PLATES (Final Solutions):\n  Plate dest_1:\n    A1: Need 5 mM Glucose, 2 mM Salt, 1 mM Phosphate\n    A2: Need 10 mM Glucose, 1 mM Salt, 0.5 mM Phosphate\n

In [117]:
# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================
# Import required libraries for data manipulation and calculations
#
# FOR JUNIOR DEVELOPERS:
# These are external libraries (not built into Python) that provide powerful tools:
# - pandas: Like Excel for Python - read/write CSV files, work with tables
# - numpy: Math library - handles numbers, arrays, and scientific calculations
# - os: Operating system interface - check if files exist, work with file paths
# - defaultdict: Special dictionary that auto-creates missing keys (useful for counting)

import pandas as pd      # For reading CSV files and working with DataFrames (tables)
                          # DataFrame = a table with rows and columns (like Excel spreadsheet)
                          # Example: df['Component'] gives you a column, df.loc[0] gives you a row

import numpy as np       # For numerical calculations and comparisons
                          # Example: np.isclose(0.1 + 0.2, 0.3) checks if two floats are "equal"
                          # (Computers can't represent 0.1 + 0.2 exactly as 0.3 due to binary math)

import os                # For file path operations
                          # Example: os.path.exists('file.csv') checks if a file exists

from collections import defaultdict  # For tracking source well usage
                                      # Example: d = defaultdict(float) creates a dict that defaults to 0.0
                                      # d['key'] += 5 works even if 'key' doesn't exist yet

In [118]:
# =============================================================================
# SECTION 2: USER PARAMETERS
# =============================================================================
# This section defines all the configuration settings you can adjust.
# These parameters control how the script behaves and what files it uses.

import math  # For ceiling function (rounding up) when splitting large transfers

# Dictionary containing all user-configurable parameters
user_params = {
    # ========================================================================
    # INPUT FILES
    # ========================================================================
    # Paths to CSV files containing input data
    'stock_conc_file': 'data/stock_concentrations_REE.csv',           # List of components and their concentrations
    'stock_plate_high_file': 'data/24-well_stock_plate_high.csv', # High-concentration stock plate layout
    'stock_plate_low_file': 'data/24-well_stock_plate_low.csv',   # Low-concentration stock plate layout
    'target_conc_file': 'data/target_concentrations.csv',         # Target concentrations for each well

    # ========================================================================
    # LIQUID HANDLING PARAMETERS
    # ========================================================================
    # These parameters define the physical constraints of the liquid handler
    # 
    # IMPORTANT FOR JUNIOR DEVELOPERS:
    # These values are based on real lab equipment limitations. Changing them
    # without understanding the hardware can cause the liquid handler to fail!
    
    'well_volume': 1000,            # Total volume (µL) in each destination well after all transfers
                                     # Example: If you add 10µL Glucose + 5µL Salt + 1485µL Water = 1500µL total
                                     # This is the final volume in each destination well
    
    'source_well_volume': 9000,      # Maximum usable volume (µL) in each source well
                                     # Example: A source well can hold 9000µL of stock solution
                                     # This is the TOTAL volume, but we can't use all of it (see dead_volume)
    
    'dead_volume': 100,              # Volume (µL) that cannot be aspirated from source wells
                                     # WHY: The pipette tip can't reach the very bottom of the well
                                     # Example: Well has 9000µL total, but only 8900µL is usable (9000 - 100)
                                     # This prevents the pipette from running dry during transfers
    
    'min_transfer_volume': 5.0,      # Minimum accurate transfer volume (µL)
                                     # WHY: Liquid handlers are less accurate with very small volumes
                                     # Example: Transferring 2µL might be inaccurate, but 5µL is reliable
                                     # If calculation requires < 5µL, we'll try a different stock or skip it
    
    'max_tip_volume': 200.0,         # Maximum tip volume (µL) - physical limit of pipette tip
                                     # WHY: Pipette tips have a maximum capacity
                                     # Example: If we need to transfer 250µL, we split it into:
                                     #   - Cycle 1: 125µL
                                     #   - Cycle 2: 125µL
                                     # This ensures we don't exceed the tip's physical capacity
    
    'culture_factor': 100,           # Culture dilution factor (unitless ratio)
                                     # HOW IT WORKS: culture_vol = well_volume / culture_factor
                                     # Example: 1500µL / 100 = 15µL culture per well
                                     # This creates a 100x dilution (1 part culture, 99 parts other)
                                     # WHY: Cultures are added by volume ratio, not concentration
    
    'epsilon': 1e-6,                 # Floating point comparison tolerance (very small number)
                                     # WHY: Computers can't represent decimals perfectly
                                     # Example: 0.1 + 0.2 might equal 0.3000000001 instead of 0.3
                                     # We use epsilon to check if two floats are "close enough"
                                     # Instead of: if x == y, we do: if abs(x - y) < epsilon

    # ========================================================================
    # PLATE FORMAT
    # ========================================================================
    # Configuration for destination plate format
    'wells_per_plate': 96,         # Number of wells per destination plate (48 or 96)
    'plate_format': '96-well',      # Plate format: '48-well' (A1-F8) or '96-well' (A1-H12)

    # ========================================================================
    # CULTURE HANDLING
    # ========================================================================
    # Controls whether culture is added to all wells or only specific ones
    'add_culture_to_all_wells': False,  # If True: add culture to all wells
                                       # If False: only add where target_conc['Culture'] > 0

    # ========================================================================
    # FIXED BASE VOLUME
    # ========================================================================
    # Fixed constant volume (e.g., base media, antibiotics) pre-loaded in wells
    # This volume is added manually by user (external workflow)
    # Script accounts for it in water calculation
    # Set to 0.0 to disable (default, backward compatible)
    'fixed_base_volume': 210.0,  # µL - reserved space, not included in transfer instructions
        # 10 ul Nd ; 100ul culture ; 100ul 10X buffer = 210
    # ========================================================================
    # WATER SOURCE CONFIGURATION
    # ========================================================================
    # Where to get water from (used to fill wells to target volume)
    'water_source': {
        'plate': 's_water',  # Plate name (e.g., 's_water') or 'reservoir_1', 'trough_1', etc.
        'well': 'none',        # Well name (None for reservoir/trough - unlimited source)
        'type': 'reservoir'      # Source type: 'plate', 'reservoir', or 'trough'
    },

    # ========================================================================
    # OUTPUT
    # ========================================================================
    # Where to save the final transfer instructions
    'output_file': 'data/transfer_instructions.csv'
}

# Print loaded parameters for verification
print("User parameters loaded:")
for key, value in user_params.items():
    if key not in ['water_source']:  # Skip complex objects (dictionaries)
        print(f"  {key}: {value}")

User parameters loaded:
  stock_conc_file: data/stock_concentrations_REE.csv
  stock_plate_high_file: data/24-well_stock_plate_high.csv
  stock_plate_low_file: data/24-well_stock_plate_low.csv
  target_conc_file: data/target_concentrations.csv
  well_volume: 1000
  source_well_volume: 9000
  dead_volume: 100
  min_transfer_volume: 5.0
  max_tip_volume: 200.0
  culture_factor: 100
  epsilon: 1e-06
  wells_per_plate: 96
  plate_format: 96-well
  add_culture_to_all_wells: False
  fixed_base_volume: 210.0
  output_file: data/transfer_instructions.csv


In [119]:
# =============================================================================
# SECTION 3: LOAD INPUT DATA
# =============================================================================
# This section reads all the input CSV files and loads them into pandas DataFrames.
# DataFrames are like Excel spreadsheets - tables with rows and columns.

# -----------------------------------------------------------------------------
# Load stock concentrations file
# -----------------------------------------------------------------------------
# This file contains a list of all components and their stock concentrations.
# Format: Component (index), Concentration[mM] (column)
df_stock = pd.read_csv(user_params['stock_conc_file'])
df_stock = df_stock.set_index('Component')  # Use Component column as row index
                                             # Makes it easy to look up by component name

# -----------------------------------------------------------------------------
# Load stock plate layout files
# -----------------------------------------------------------------------------
# These files tell us WHERE each component is located on the stock plates.
# Format: Component, Well, Concentration[mM]
df_stock_plate_high = pd.read_csv(user_params['stock_plate_high_file'])  # High-concentration plate (s1)
df_stock_plate_low = pd.read_csv(user_params['stock_plate_low_file'])    # Low-concentration plate (s4)

# -----------------------------------------------------------------------------
# Load target concentrations file
# -----------------------------------------------------------------------------
# This file specifies what concentration of each component we want in each destination well.
# Format: Well (index), Component1, Component2, ... (columns)
# Each cell contains the target concentration (mM) for that component in that well
df_target_conc = pd.read_csv(user_params['target_conc_file'], index_col=0)  # First column is well names

# Print summary of loaded data
print("Data loaded successfully:")
print(f"  Stock components: {len(df_stock)}")  # Number of different components
print(f"  High plate wells: {len(df_stock_plate_high)}")  # Number of wells in high plate
print(f"  Low plate wells: {len(df_stock_plate_low)}")    # Number of wells in low plate
print(f"  Target wells: {len(df_target_conc)}")            # Number of destination wells
print(f"  Target components: {len(df_target_conc.columns)}")  # Number of components to add

df_stock_plate_high
df_stock_plate_low
df_target_conc
df_stock_plate_high

Data loaded successfully:
  Stock components: 6
  High plate wells: 6
  Low plate wells: 6
  Target wells: 139
  Target components: 6


,Well,Component,Concentration
0,A1,Phosphates,500.00
1,B1,NH4SO4,1000.00
2,C1,CoCl2,0.60
3,A2,Succinate,600.00
4,B2,Methanol,3000.00
5,C2,PQQ,0.05


In [120]:
# =============================================================================
# SECTION 4: BUILD STOCK LOOKUP DICTIONARY
# =============================================================================
# PURPOSE:
# This section creates a lookup table (dictionary) that maps each component
# to its source location(s) on the stock plates.
#
# WHAT IT DOES:
# For each component (e.g., "Glucose"), it tells us:
#   - Where is it located? (which plate and well)
#   - What concentration is it? (in mM)
#   - Is it available in high or low concentration?
#
# DATA STRUCTURE:
# stock_lookup[component_name] = {
#     'high': [  # List of wells (supports multiple wells per component)
#         {'plate': 's1', 'well': 'A1', 'conc': 100.0},
#         {'plate': 's1', 'well': 'A2', 'conc': 100.0}  # If multiple wells exist
#     ] or None,  # None means no high-concentration stock available
#     'low': [
#         {'plate': 's4', 'well': 'B2', 'conc': 10.0}
#     ] or None
# }
#
# EXAMPLE:
# stock_lookup['Glucose'] = {
#     'high': [{'plate': 's1', 'well': 'A1', 'conc': 100.0}],
#     'low': [{'plate': 's4', 'well': 'B2', 'conc': 10.0}]
# }
# This means: Glucose is available at 100 mM in s1:A1 and 10 mM in s4:B2
#
# WHY MULTIPLE WELLS?
# If a component appears in multiple wells (e.g., Glucose in A1 and A2),
# we can automatically switch to the second well if the first one runs out
# (depletion prevention - see Phase 2)

stock_lookup = {}

# -----------------------------------------------------------------------------
# Step 1: Initialize all components from stock concentrations file
# -----------------------------------------------------------------------------
# Create an entry for each component, initially with no source locations
for comp in df_stock.index:
    stock_lookup[comp] = {'high': None, 'low': None}  # None = no source available

# -----------------------------------------------------------------------------
# Step 2: Populate HIGH stock sources
# -----------------------------------------------------------------------------
# Fill in the 'high' entry for each component that exists on the high plate
for _, row in df_stock_plate_high.iterrows():
    comp = row['Component']  # Component name (e.g., 'Glucose')

    # If component not in lookup yet, add it
    if comp not in stock_lookup:
        stock_lookup[comp] = {'high': None, 'low': None}

    # Initialize as empty list if None (first well for this component)
    if stock_lookup[comp]['high'] is None:
        stock_lookup[comp]['high'] = []

    # Append location and concentration (instead of overwriting)
    # This preserves all wells if component appears multiple times
    stock_lookup[comp]['high'].append({
        'plate': 's1',                    # High concentration plate is always 's1'
        'well': row['Well'],              # Well location (e.g., 'A1', 'B3')
        'conc': row['Concentration']  # Stock concentration in mM
    })

# -----------------------------------------------------------------------------
# Step 3: Populate LOW stock sources
# -----------------------------------------------------------------------------
# Fill in the 'low' entry for each component that exists on the low plate
for _, row in df_stock_plate_low.iterrows():
    comp = row['Component']

    # If component not in lookup yet, add it
    if comp not in stock_lookup:
        stock_lookup[comp] = {'high': None, 'low': None}

    # Initialize as empty list if None (first well for this component)
    if stock_lookup[comp]['low'] is None:
        stock_lookup[comp]['low'] = []

    # Append location and concentration (instead of overwriting)
    # This preserves all wells if component appears multiple times
    stock_lookup[comp]['low'].append({
        'plate': 's4',                    # Low concentration plate is always 's4'
        'well': row['Well'],
        'conc': row['Concentration']
    })

# Print summary
print(f"\nStock lookup created: {len(stock_lookup)} components")
print(f"  Components with HIGH stock: {sum(1 for c in stock_lookup.values() if c['high'] is not None)}")
print(f"  Components with LOW stock: {sum(1 for c in stock_lookup.values() if c['low'] is not None)}")

# Print components with multiple wells (fallback strategy enabled)
duplicates_found = {}
for comp, stocks in stock_lookup.items():
    for stock_level in ['high', 'low']:
        wells = stocks.get(stock_level)
        if isinstance(wells, list) and len(wells) > 1:
            duplicates_found[f"{comp}_{stock_level}"] = len(wells)

if duplicates_found:
    print(f"\nComponents with multiple wells (fallback strategy enabled):")
    for comp_level, count in sorted(duplicates_found.items()):
        print(f"  - {comp_level}: {count} wells")

# -----------------------------------------------------------------------------
# Step 4: Special handling for Culture
# -----------------------------------------------------------------------------
# Culture is handled differently - it uses volume ratio, not concentration.
# If Culture isn't in the stock plates, add it with a default source.
if 'Culture' not in stock_lookup:
    stock_lookup['Culture'] = {
        'high': [  # Now stored as list for consistency
            {
                'plate': 's1',
                'well': 'A1',  # Default source for culture
                'conc': 1.0    # Dummy concentration (Culture uses volume ratio, not concentration)
            }
        ],
        'low': None
    }
    print(f"  Added Culture to stock_lookup with default source s1:A1")


# =============================================================================
# Step 5: Build Source Inventory (Multi-Well Support)
# =============================================================================
# Create an inventory of all available source wells with their capacities.
# This supports multiple wells per component for automatic fallback when depleted.

def build_source_inventory(stock_lookup, source_well_volume, dead_volume):
    """
    Build inventory of all available source wells with their capacities.

    Args:
        stock_lookup: Dictionary mapping components to their stock locations
        source_well_volume: Total volume per source well (µL)
        dead_volume: Unaspirated volume (µL)

    Returns:
        source_inventory: {(comp, stock_level): [list of well dicts]}
        Each well dict includes: plate, well, conc, available_vol, index
    """
    source_inventory = {}
    usable_vol = source_well_volume - dead_volume

    for comp, stocks in stock_lookup.items():
        for stock_level in ['high', 'low']:
            stock_list = stocks.get(stock_level)

            if stock_list is None:
                continue

            # Handle backward compatibility: if single dict (old format), convert to list
            if isinstance(stock_list, dict):
                stock_list = [stock_list]

            key = (comp, stock_level)
            source_inventory[key] = []

            for index, stock_info in enumerate(stock_list):
                source_inventory[key].append({
                    'plate': stock_info['plate'],
                    'well': stock_info['well'],
                    'conc': stock_info['conc'],
                    'available_vol': usable_vol,
                    'index': index  # Track position for deterministic ordering
                })

    return source_inventory


Stock lookup created: 6 components
  Components with HIGH stock: 6
  Components with LOW stock: 6
  Added Culture to stock_lookup with default source s1:A1


In [121]:
# =============================================================================
# SECTION 5: VALIDATION CHECKS
# =============================================================================
# Before calculating transfers, we check that the input data is valid.
# This helps catch errors early and provides helpful error messages.

errors = []        # Critical errors that will prevent the script from working
warnings_list = [] # Warnings about potential issues (script can continue)

# -----------------------------------------------------------------------------
# Check 1: All components in target exist in stock plates
# -----------------------------------------------------------------------------
# We can't create a solution if we don't have the stock!
missing_components = []
for comp in df_target_conc.columns:
    # Check if component exists in stock lookup
    if comp not in stock_lookup:
        missing_components.append(comp)
    else:
        # Check if component has at least one stock source (high or low)
        has_source = (stock_lookup[comp]['high'] is not None or 
                     stock_lookup[comp]['low'] is not None)
        if not has_source:
            missing_components.append(comp)

if missing_components:
    errors.append(f"Components in target but not in stock plates: {missing_components}")

# -----------------------------------------------------------------------------
# Check 2: Well formats are valid
# -----------------------------------------------------------------------------
# Wells should be in format: letter + number (e.g., 'A1', 'B12', 'H8')
def is_valid_well(well):
    """Check if well format is valid (e.g., A1, B12)."""
    if pd.isna(well):  # Check for NaN/None
        return False
    well_str = str(well).strip()  # Convert to string and remove whitespace
    if len(well_str) < 2:  # Must have at least letter + number
        return False
    # First character should be a letter, rest should be digits
    return well_str[0].isalpha() and well_str[1:].isdigit()

# Check well formats in both stock plates
for plate_df, plate_name in [(df_stock_plate_high, 'high'), 
                              (df_stock_plate_low, 'low')]:
    invalid_wells = plate_df[~plate_df['Well'].apply(is_valid_well)]['Well'].tolist()
    if invalid_wells:
        warnings_list.append(f"Invalid well formats in {plate_name} plate: {invalid_wells}")

# -----------------------------------------------------------------------------
# Check 3: Concentrations are positive
# -----------------------------------------------------------------------------
# Negative concentrations don't make physical sense
for comp in df_target_conc.columns:
    negative = df_target_conc[comp][df_target_conc[comp] < 0]
    if len(negative) > 0:
        warnings_list.append(f"Negative concentrations for {comp} in wells: {negative.index.tolist()}")

# -----------------------------------------------------------------------------
# Check 4: Stock concentration > target concentration (dilution possible)
# -----------------------------------------------------------------------------
# PHYSICAL CONSTRAINT: You can only DILUTE a stock, not concentrate it
# To dilute a stock, the stock MUST be more concentrated than the target
#
# WHY THIS MATTERS:
# - If target >= stock, we can't achieve it by dilution alone
# - Example: Can't make 100 mM from 50 mM stock (would need to concentrate, not dilute)
# - Example: Can make 5 mM from 100 mM stock (dilution works)
#
# NUMERICAL EXAMPLES:
# VALID (can dilute):
#   Stock: 100 mM, Target: 5 mM → ✓ Can dilute (100 > 5)
#   Stock: 10 mM, Target: 1 mM → ✓ Can dilute (10 > 1)
#
# INVALID (cannot dilute):
#   Stock: 50 mM, Target: 100 mM → ✗ Cannot achieve (50 < 100)
#   Stock: 5 mM, Target: 5 mM → ✗ Cannot achieve (5 = 5, no dilution)
#
# WHAT WE CHECK:
# For each component, we find:
#   - max_target: Highest target concentration requested
#   - max_stock: Highest stock concentration available
# If max_target >= max_stock, we have a problem!
for comp in df_target_conc.columns:
    if comp in stock_lookup:
        max_target = df_target_conc[comp].max()  # Highest target concentration for this component
        
        # Find the maximum stock concentration available (from high or low)
        max_stock = 0
        for stock_type in ['high', 'low']:
            stock_list = stock_lookup[comp][stock_type]
            if stock_list is not None:
                # Handle both old (dict) and new (list) formats
                wells = stock_list if isinstance(stock_list, list) else [stock_list]
                for well_info in wells:
                    conc = well_info.get('conc')
                    if conc is not None and conc > max_stock:
                        max_stock = conc

        # If target is >= stock, we can't dilute to achieve it
        if max_stock > 0 and max_target >= max_stock:
            errors.append(f"Component {comp}: max target ({max_target}) >= max stock ({max_stock}) - cannot dilute")

# -----------------------------------------------------------------------------
# Check 5: Fixed base volume configuration
# -----------------------------------------------------------------------------
# Validate fixed_base_volume parameter
fixed_base_vol = user_params.get('fixed_base_volume', 0)
if fixed_base_vol < 0:
    errors.append(f"fixed_base_volume cannot be negative: {fixed_base_vol}")

if fixed_base_vol >= user_params['well_volume']:
    errors.append(
        f"fixed_base_volume ({fixed_base_vol} µL) >= well_volume ({user_params['well_volume']} µL) - "
        f"no space for components"
    )

# Warning: very small effective space for components
if 0 < fixed_base_vol < user_params['well_volume']:
    effective_space = user_params['well_volume'] - fixed_base_vol
    if effective_space < 100:
        warnings_list.append(
            f"Effective space for components very small ({effective_space} µL) - "
            f"may cause issues with calculations"
        )

# -----------------------------------------------------------------------------
# Print validation results
# -----------------------------------------------------------------------------
print("Validation complete:")
if errors:
    print(f"  ERRORS ({len(errors)}):")
    for err in errors:
        print(f"    - {err}")
else:
    print("  No errors found")

if warnings_list:
    print(f"  WARNINGS ({len(warnings_list)}):")
    for warn in warnings_list[:5]:  # Show first 5 warnings
        print(f"    - {warn}")
    if len(warnings_list) > 5:
        print(f"    ... and {len(warnings_list) - 5} more warnings")
else:
    print("  No warnings")

Validation complete:
  No errors found
  No warnings


In [122]:
# =============================================================================
# SECTION 6: CORE ALGORITHM - find_volumes_bulk FUNCTION
# =============================================================================
# This is the heart of the script! This function calculates how much volume
# of each component needs to be transferred to each destination well.

def find_volumes_bulk(df_stock, df_target_conc, well_volume, min_tip_volume, culture_ratio, stock_lookup,
                      epsilon=1e-6, add_culture_to_all=True, fixed_base_volume=0.0):
    """
    Calculate transfer volumes for all wells and components.
    
    THIS IS THE CORE ALGORITHM - THE HEART OF THE SCRIPT!
    
    PURPOSE:
    For each destination well and each component, calculate exactly how much
    volume needs to be transferred from stock solutions to achieve target concentrations.
    
    COMPLETE WALKTHROUGH EXAMPLE:
    -----------------------------
    Let's say we want to make a solution in well A1 with:
      - 5 mM Glucose
      - 2 mM Salt
      - 1 mM Phosphate
      - Culture (100x dilution)
      - Total volume: 1500 µL
    
    Step 1: Process Glucose
      - Target: 5 mM in 1500 µL
      - Available stocks: 100 mM (high) or 10 mM (low)
      - Try HIGH first: transfer_vol = (5 × 1500) / 100 = 75 µL
      - Check: 75 µL >= 5 µL minimum? YES ✓
      - Use HIGH stock: 75 µL from s1:A1
    
    Step 2: Process Salt
      - Target: 2 mM in 1500 µL
      - Available stocks: 50 mM (high) or 5 mM (low)
      - Try HIGH first: transfer_vol = (2 × 1500) / 50 = 60 µL
      - Check: 60 µL >= 5 µL? YES ✓
      - Use HIGH stock: 60 µL from s1:B2
    
    Step 3: Process Phosphate
      - Target: 1 mM in 1500 µL
      - Available stocks: 100 mM (high) or 10 mM (low)
      - Try HIGH first: transfer_vol = (1 × 1500) / 100 = 15 µL
      - Check: 15 µL >= 5 µL? YES ✓
      - Use HIGH stock: 15 µL from s1:C3
    
    Step 4: Process Culture
      - Culture uses volume ratio, not concentration
      - culture_vol = well_volume / culture_factor = 1500 / 100 = 15 µL
      - Use: 15 µL from s1:A1 (culture source)
    
    Step 5: Calculate Water
      - Sum of components: 75 + 60 + 15 + 15 = 165 µL
      - Water needed: 1500 - 165 = 1335 µL
      - Use: 1335 µL from s_water:A1
    
    Final Result for Well A1:
      - Glucose: 75 µL from s1:A1
      - Salt: 60 µL from s1:B2
      - Phosphate: 15 µL from s1:C3
      - Culture: 15 µL from s1:A1
      - Water: 1335 µL from s_water:A1
      - Total: 1500 µL ✓

    HOW IT WORKS (ALGORITHM):
    For each well and each component:
    1. Get target concentration
    2. Find appropriate stock (high or low concentration)
    3. Calculate: transfer_vol = (target_conc × well_volume) / stock_conc
    4. Check if volume meets minimum requirement
    5. Add water to fill well to target volume
    6. Account for any fixed_base_volume in water calculation

    Args:
        df_stock: DataFrame with stock concentrations
        df_target_conc: DataFrame with target concentrations (Well × Component)
        well_volume: Total volume (µL) in each destination well
        min_tip_volume: Minimum accurate transfer volume (µL)
        culture_ratio: Culture dilution factor (e.g., 100 for 100x dilution)
        stock_lookup: Dictionary mapping components to their source locations
        epsilon: Floating point comparison tolerance
        add_culture_to_all: If False, only add culture where target_conc['Culture'] > 0
        fixed_base_volume: Fixed constant volume (µL) pre-loaded in wells (default: 0.0)

    Returns:
        df_volumes: DataFrame (Well × Component) with transfer volumes in µL
        df_conc_level: DataFrame (Well × Component) with stock level used ('high', 'low', 'fresh')
        errors: List of error messages
        warnings_list: List of warning messages
    """
    
    # Initialize output DataFrames
    # These will store the calculated volumes and which stock was used
    df_volumes = pd.DataFrame(index=df_target_conc.index, columns=df_target_conc.columns)
    df_conc_level = pd.DataFrame(index=df_target_conc.index, columns=df_target_conc.columns)

    errors = []
    warnings_list = []

    # -------------------------------------------------------------------------
    # Process each well and each component
    # -------------------------------------------------------------------------
    for well in df_target_conc.index:  # Loop through each destination well
        for comp in df_target_conc.columns:  # Loop through each component
            target_conc = df_target_conc.loc[well, comp]  # Get target concentration

            # Skip if zero, None, or NaN (using epsilon for floating point comparison)
            # We don't need to transfer anything if target is zero
            if pd.isna(target_conc) or abs(target_conc) < epsilon:
                df_volumes.loc[well, comp] = 0.0
                df_conc_level.loc[well, comp] = None
                if pd.isna(target_conc):
                    warnings_list.append(f"Well {well}, {comp}: NaN concentration, skipping")
                continue

            # Check if component exists in stock lookup
            if comp not in stock_lookup:
                errors.append(f"Well {well}, {comp}: Component not in stock lookup")
                df_volumes.loc[well, comp] = 0.0
                df_conc_level.loc[well, comp] = None
                continue

            stocks = stock_lookup[comp]  # Get available stocks for this component
            transfer_vol = None  # Will store calculated transfer volume
            stock_level = None   # Will store which stock we used ('high' or 'low')

            # -----------------------------------------------------------------
            # Try stock types in priority order: high → low
            # -----------------------------------------------------------------
            # STOCK SELECTION STRATEGY:
            # We prefer HIGH concentration stocks first, then fall back to LOW
            #
            # WHY PREFER HIGH CONCENTRATION?
            # 1. Smaller transfer volumes = more accurate
            #    Example: To get 5 mM in 1500 µL:
            #      - High stock (100 mM): 75 µL needed
            #      - Low stock (10 mM): 750 µL needed
            #    Smaller volumes are easier to pipette accurately
            #
            # 2. Less pipetting time (fewer cycles if volume is large)
            #    Example: 750 µL with max_tip_volume=200 requires 4 cycles
            #             75 µL with max_tip_volume=200 requires 1 cycle
            #
            # 3. Less source well depletion
            #    Example: 75 µL uses less of the source well than 750 µL
            #
            # FALLBACK TO LOW:
            # If high stock would require < min_transfer_volume, we try low stock
            # Example: If high stock needs 2 µL (< 5 µL minimum), try low stock
            for stock_type in ['high', 'low']:
                stock_list = stocks[stock_type]

                # Skip if this stock type doesn't exist
                if stock_list is None:
                    continue

                # Handle both list (new format) and dict (old format) for backward compatibility
                wells = stock_list if isinstance(stock_list, list) else [stock_list]

                # -----------------------------------------------------------------
                # Special handling for Culture (uses volume ratio, not concentration)
                # -----------------------------------------------------------------
                # CULTURE IS DIFFERENT FROM OTHER COMPONENTS:
                # - Other components: Use concentration (mM) to calculate volume
                # - Culture: Use volume ratio (dilution factor) to calculate volume
                #WHY?
                # Cultures are living cells/bacteria. We add them by dilution ratio,
                # not by chemical concentration. The "concentration" doesn't matter
                # - we just want a specific dilution (e.g., 1:100).
                #
                # CALCULATION:
                #   transfer_vol = well_volume / culture_ratio
                #
                # NUMERICAL EXAMPLE:
                #   well_volume = 1500 µL
                #   culture_ratio = 100 (meaning 100x dilution)
                #   transfer_vol = 1500 / 100 = 15 µL
                #
                #   This means: Add 15 µL culture to 1500 µL total
                #   Dilution = 15 / 1500 = 1/100 = 100x dilution ✓
                #
                #   If we added 30 µL: 30 / 1500 = 1/50 = 50x dilution (wrong!)
                #   If we added 7.5 µL: 7.5 / 1500 = 1/200 = 200x dilution (wrong!)
                #
                # Note: 'fresh' is just a marker - Culture comes from high plate (s1)
                if comp == 'Culture':
                    transfer_vol = well_volume / culture_ratio  # Calculate volume based on ratio
                    stock_level = 'fresh'  # Mark as 'fresh' (special handling, not 'high' or 'low')
                    break  # Found what we need, stop looking (no need to check other stocks)

                # -----------------------------------------------------------------
                # Regular components: calculate transfer volume from concentration
                # -----------------------------------------------------------------
                # Try each available well for this stock type (usually just one, but may be multiple)
                found_valid_stock = False
                for stock_info in wells:
                    # Get stock concentration
                    stock_conc = stock_info.get('conc')
                    if stock_conc is None:
                        continue

                    # Handle "300x" format (shouldn't happen after stock lookup, but just in case)
                    # Some files might have concentrations like "300x" instead of a number
                    if isinstance(stock_conc, str) and 'x' in str(stock_conc).lower():
                        try:
                            factor = float(str(stock_conc).replace('x', '').replace('X', '').strip())
                            stock_conc = factor
                        except (ValueError, TypeError):
                            continue  # Skip this stock if we can't parse it

                    # -----------------------------------------------------------------
                    # Calculate transfer volume using dilution formula
                    # -----------------------------------------------------------------
                    # DILUTION FORMULA (C1 × V1 = C2 × V2):
                    #   This is the fundamental equation for diluting solutions
                    #   It says: "Amount of substance before = Amount after"
                    #
                    # VARIABLES:
                    #   C1 = stock concentration (mM) - what we have in source
                    #   V1 = transfer volume (µL) - what we need to calculate (UNKNOWN)
                    #   C2 = target concentration (mM) - what we want in destination
                    #   V2 = well volume (µL) - total volume in destination (known)
                    #
                    # SOLVING FOR V1:
                    #   C1 × V1 = C2 × V2
                    #   V1 = (C2 × V2) / C1
                    #
                    # NUMERICAL EXAMPLE:
                    #   Target: 5 mM Glucose in 1500 µL well
                    #   Stock: 100 mM Glucose available
                    #   Calculation: V1 = (5 mM × 1500 µL) / 100 mM
                    #                V1 = 7500 / 100
                    #                V1 = 75 µL
                    #   Result: Transfer 75 µL of 100 mM stock → get 5 mM in 1500 µL well
                    #
                    # VERIFICATION (back-calculate):
                    #   Final concentration = (75 µL × 100 mM) / 1500 µL = 5 mM ✓
                    transfer_vol = (target_conc * well_volume) / stock_conc

                    # -----------------------------------------------------------------
                    # Check if volume meets minimum requirement
                    # -----------------------------------------------------------------
                    # If transfer volume is too small, it may be inaccurate.
                    # We check if it's >= min_tip_volume (using epsilon for floating point comparison)
                    if transfer_vol >= min_tip_volume - epsilon:
                        stock_level = stock_type  # This stock works!
                        found_valid_stock = True
                        break  # Found a valid stock, stop looking through wells

                # If we found a valid stock at this level, stop searching (don't try alternatives)
                if found_valid_stock:
                    break

            # -----------------------------------------------------------------
            # Store the result
            # -----------------------------------------------------------------
            if transfer_vol is not None and stock_level is not None:
                # We found a valid stock - store the volume and stock level
                df_volumes.loc[well, comp] = round(transfer_vol, 2)  # Round to 2 decimal places
                df_conc_level.loc[well, comp] = stock_level
            else:
                # No valid stock found - volume would be too small
                # Note this but don't error (allows small concentrations to be skipped)
                warnings_list.append(f"Well {well}, {comp}: Cannot transfer (all stocks require < {min_tip_volume} µL)")
                df_volumes.loc[well, comp] = 0.0
                df_conc_level.loc[well, comp] = None

    # -------------------------------------------------------------------------
    # Add Culture to wells (configurable)
    # -------------------------------------------------------------------------
    # Culture is handled separately because it uses volume ratio, not concentration
    if 'Culture' not in df_volumes.columns:
        if add_culture_to_all:
            # Add culture to all wells
            culture_vol = well_volume / culture_ratio  # Calculate volume based on ratio
            df_volumes['Culture'] = round(culture_vol, 2)
            df_conc_level['Culture'] = 'high'  # Culture from high plate
        else:
            # Only add culture where target_conc has Culture > 0
            if 'Culture' in df_target_conc.columns:
                for well in df_target_conc.index:
                    culture_target = df_target_conc.loc[well, 'Culture']
                    if not pd.isna(culture_target) and culture_target > epsilon:
                        culture_vol = well_volume / culture_ratio
                        df_volumes.loc[well, 'Culture'] = round(culture_vol, 2)
                        df_conc_level.loc[well, 'Culture'] = 'high'
                    else:
                        df_volumes.loc[well, 'Culture'] = 0.0
                        df_conc_level.loc[well, 'Culture'] = None
            else:
                # No Culture column in target, don't add
                pass

    # -------------------------------------------------------------------------
    # Add Fixed Base Volume (if configured)
    # -------------------------------------------------------------------------
    # Fixed base volume is pre-loaded by user (external workflow)
    # Script tracks it internally but does NOT generate transfer instructions
    # This volume is accounted for in water calculation
    if fixed_base_volume > 0:
        df_volumes['FixedBase'] = fixed_base_volume
        df_conc_level['FixedBase'] = 'fixed'  # Special marker for fixed volume

    # -------------------------------------------------------------------------
    # Calculate water volume for each well
    # -------------------------------------------------------------------------
    # WATER CALCULATION PURPOSE:
    # Water fills the remaining volume to reach the target well_volume
    # This ensures every well ends up with exactly 1500 µL total
    #
    # FORMULA: water_vol = well_volume - sum(all_component_volumes)
    #
    # NUMERICAL EXAMPLE:
    #   Target well volume: 1500 µL
    #   Component volumes:
    #     - Glucose: 75 µL
    #     - Salt: 30 µL
    #     - Culture: 15 µL
    #     - FixedBase: 200 µL (if configured)
    #   Sum of components: 75 + 30 + 15 + 200 = 320 µL
    #   Water needed: 1500 - 320 = 1180 µL
    #
    # WHY THIS MATTERS:
    # - Ensures consistent total volume across all wells
    # - Accounts for all components including FixedBase
    # - If FixedBase is 200 µL, water automatically reduces by 200 µL
    #
    # Note: If FixedBase is present, it's included in the sum, so water auto-adjusts
    df_volumes['Water'] = 0.0
    for well in df_volumes.index:
        # Sum all component volumes (excluding Water itself to avoid circular calculation)
        total_vol = df_volumes.loc[well, df_volumes.columns != 'Water'].sum()
        water_vol = well_volume - total_vol

        # Use epsilon for floating point comparison
        if water_vol < -epsilon:
            # Error: total volume exceeds well capacity (shouldn't happen)
            errors.append(f"Well {well}: Total volume {total_vol:.2f} exceeds {well_volume} µL")
            water_vol = 0.0
        elif abs(water_vol) < epsilon:
            water_vol = 0.0  # Snap to zero if very small (rounding error)
        elif water_vol > 0 and water_vol < min_tip_volume - epsilon:
            # Warning: water volume is very small (may be inaccurate)
            warnings_list.append(f"Well {well}: Water volume {water_vol:.2f} < {min_tip_volume} µL (may be inaccurate)")

        df_volumes.loc[well, 'Water'] = round(water_vol, 2)

    return df_volumes, df_conc_level, errors, warnings_list

print("find_volumes_bulk function defined")

find_volumes_bulk function defined


In [137]:
# =============================================================================
# SECTION 7: RUN VOLUME CALCULATIONS
# =============================================================================
# Execute the find_volumes_bulk function to calculate all transfer volumes.

df_volumes, df_conc_level, calc_errors, calc_warnings = find_volumes_bulk(
    df_stock=df_stock,
    df_target_conc=df_target_conc,
    well_volume=user_params['well_volume'],
    min_tip_volume=user_params['min_transfer_volume'],
    culture_ratio=user_params['culture_factor'],
    stock_lookup=stock_lookup,
    epsilon=user_params['epsilon'],
    add_culture_to_all=user_params['add_culture_to_all_wells'],
    fixed_base_volume=user_params.get('fixed_base_volume', 0.0)
)

print("Volume calculations complete!")
print(f"\nVolumes dataframe shape: {df_volumes.shape}")  # (number of wells, number of components)
print(f"Conc level dataframe shape: {df_conc_level.shape}")

# Print any errors or warnings from the calculation
if calc_errors:
    print(f"\nCalculation ERRORS ({len(calc_errors)}):")
    for err in calc_errors:  # Show first 10
        print(f"  - {err}")
    

if calc_warnings:
    print(f"\nCalculation WARNINGS ({len(calc_warnings)}):")
    for warn in calc_warnings:  # Show first 10
        print(f"  - {warn}")

# Show sample results
print("\nSample volumes (first well, first 5 components):")
print(df_volumes.iloc[0, :5])
print("\nSample conc levels:")
print(df_conc_level.iloc[0, :5])

Volume calculations complete!

Volumes dataframe shape: (139, 8)
Conc level dataframe shape: (139, 7)

Calculation WARNINGS (32):
  - Well A3, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well A7, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well A11, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well B5, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well B6, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well B12, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well C5, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well C11, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well D4, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well D6, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well F5, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well G2, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well G7, CoCl2: Cannot transfer (all stocks require < 5.0 µL)
  - Well 

In [124]:
# =============================================================================
# SECTION 8: WELL REMAPPING FOR DIFFERENT PLATE FORMATS
# =============================================================================
# This section handles different plate formats (48-well vs 96-well).
# It remaps well IDs if needed and assigns destination wells to plates.

# -----------------------------------------------------------------------------
# Function: remap_well_for_plate
# -----------------------------------------------------------------------------
def remap_well_for_plate(well, plate_format='48-well'):
    """
    Remap well IDs for different plate formats.
    
    Converts 96-well format (A1-H12) to 48-well format (A1-F8) if needed.
    
    Plate formats:
    - 48-well: 6 rows (A-F) × 8 columns (1-8) = 48 wells
    - 96-well: 8 rows (A-H) × 12 columns (1-12) = 96 wells
    
    Args:
        well: Well ID (e.g., 'A1', 'H12')
        plate_format: '48-well' or '96-well'
    
    Returns:
        Remapped well ID or original if no remapping needed
    """
    if pd.isna(well):
        return None

    well_str = str(well).strip().upper()  # Convert to uppercase string
    if len(well_str) < 2:
        return well_str

    row = well_str[0]      # Extract row letter (A, B, C, ...)
    col = int(well_str[1:])  # Extract column number (1, 2, 3, ...)

    # If 48-well format, validate well exists
    if plate_format == '48-well':
        # 48-well: 6 rows (A-F) × 8 columns (1-8)
        max_row = 'F'
        max_col = 8

        # If well is beyond 48-well format, remap or error
        if row > max_row or col > max_col:
            # Try to remap: G1-H12 -> F1-F8 (last row)
            if row > max_row:
                row = max_row
            if col > max_col:
                col = max_col
            return f"{row}{col}"

    # 96-well format: 8 rows (A-H) × 12 columns (1-12) - no remapping needed
    return well_str


# -----------------------------------------------------------------------------
# Function: assign_dest_plates
# -----------------------------------------------------------------------------
def assign_dest_plates(wells, wells_per_plate=48):
    """
    Assign destination wells to plates.
    
    This function splits wells across multiple destination plates.
    For example, if you have 100 wells and wells_per_plate=48:
    - Wells 1-48 → dest_1
    - Wells 49-96 → dest_2
    - Wells 97-100 → dest_3
    
    Args:
        wells: List of well IDs
        wells_per_plate: Number of wells per plate (48 or 96)
    
    Returns:
        Dictionary mapping well → plate name (e.g., {'A1': 'dest_1', 'A2': 'dest_1', ...})
    """
    plate_num = 1
    well_count = 0
    assignments = {}

    for well in wells:
        # If we've filled a plate, move to the next one
        if well_count >= wells_per_plate:
            plate_num += 1
            well_count = 0

        dest_plate = f"dest_{plate_num}"  # Create plate name (e.g., 'dest_1', 'dest_2')
        assignments[well] = dest_plate
        well_count += 1

    return assignments

# -----------------------------------------------------------------------------
# Assign destination plates
# -----------------------------------------------------------------------------
# Assign each destination well to a plate
dest_plate_assignments = assign_dest_plates(
    df_target_conc.index,  # List of all destination well IDs
    wells_per_plate=user_params['wells_per_plate']
)

# Remap wells if needed (for 48-well format)
plate_format = user_params.get('plate_format', '48-well')
if plate_format == '48-well':
    # Remap destination wells to 48-well format
    remapped_assignments = {}
    for well, plate in dest_plate_assignments.items():
        remapped_well = remap_well_for_plate(well, plate_format)
        remapped_assignments[remapped_well] = plate
    dest_plate_assignments = remapped_assignments

print(f"Assigned {len(dest_plate_assignments)} wells to destination plates")
print(f"Number of destination plates: {max([int(p.split('_')[1]) for p in dest_plate_assignments.values()])}")

Assigned 139 wells to destination plates
Number of destination plates: 2


In [125]:
# =============================================================================
# SECTION 9: BUILD TRANSFER RECORDS
# =============================================================================
# This section converts the calculated volumes into transfer instructions.
# Each transfer record tells the liquid handler: move X µL from source to destination.

transfers = []  # List to store all transfer records
source_usage = defaultdict(float)  # Track cumulative volume usage: {(plate, well): volume}
                                     # Used to check if source wells will be depleted
transfer_errors = []
transfer_warnings = []

# =============================================================================
# PHASE 2 INITIALIZATION: Build Source Inventory for Smart Selection
# =============================================================================
# Build inventory of all available source wells with capacity tracking
# This enables automatic fallback when wells deplete
source_inventory = build_source_inventory(
    stock_lookup,
    user_params['source_well_volume'],
    user_params['dead_volume']
)
usable_vol = user_params['source_well_volume'] - user_params['dead_volume']

# Tracking variables for Phase 2
well_switches = []         # Log automatic well switches due to depletion
failed_transfers = []      # Log transfers that couldn't be assigned

# -----------------------------------------------------------------------------
# Helper function: get_source_well
# -----------------------------------------------------------------------------
def get_source_well(comp, stock_level, stock_lookup, df_stock_plate_high, df_stock_plate_low):
    """
    Get source plate and well for a component based on stock level.
    Now supports multiple wells per component (stored as list).

    NOTE: This is a compatibility function for Phase 1. Phase 2 will replace
    this with get_available_source_well() for smart selection with depletion.

    Args:
        comp: Component name
        stock_level: 'high', 'low', or 'fresh' (for Culture)
        stock_lookup: Dictionary mapping components to source locations (now lists)

    Returns:
        (source_plate, source_well) tuple, or (None, None) if not found
    """
    if comp not in stock_lookup:
        return None, None

    stocks = stock_lookup[comp]

    # Special handling for Culture (from high plate)
    if comp == 'Culture':
        stock_list = stocks.get('high')
        if stock_list is not None:
            # Handle both list and single dict formats
            if isinstance(stock_list, list) and len(stock_list) > 0:
                well_info = stock_list[0]  # Use first well for now
                return well_info['plate'], well_info['well']
            elif isinstance(stock_list, dict):
                return stock_list['plate'], stock_list['well']
        return 's1', 'A1'  # Default (high plate)

    # Regular components: use stock_level to determine plate
    stock_list = stocks.get(stock_level)
    if stock_list is not None:
        # Handle both list and single dict formats
        if isinstance(stock_list, list) and len(stock_list) > 0:
            well_info = stock_list[0]  # Use first well for now (Phase 2 will improve this)
            return well_info['plate'], well_info['well']
        elif isinstance(stock_list, dict):
            return stock_list['plate'], stock_list['well']

    return None, None


# =============================================================================
# PHASE 2: Smart Well Selection with Depletion Prevention
# =============================================================================
# This function implements automatic fallback when source wells deplete.
# It checks capacity BEFORE assignment to prevent invalid transfers.

def get_available_source_well(comp, stock_level, source_inventory, source_usage,
                              transfer_vol, usable_vol):
    """
    Get an available source well for a component with smart fallback strategy.

    This replaces get_source_well() for Phase 2+ to provide:
    1. Real-time capacity checking
    2. Automatic fallback to alternative stock level
    3. Deterministic ordering (file order, not "best capacity")
    4. Clear tracking of well switches

    Strategy (in order):
    1. Try primary stock_level: find first well with sufficient capacity
    2. Try alternative stock_level: same search
    3. Return None if no wells available

    Special cases:
    - Culture: Always attempts primary source first, no fallback
    - Volume constraints checked BEFORE assignment

    Args:
        comp: Component name (e.g., 'Glucose', 'Culture')
        stock_level: 'high' or 'low'
        source_inventory: {(comp, stock_level): [list of well dicts]}
        source_usage: {(plate, well): cumulative_volume}
        transfer_vol: Required transfer volume in µL
        usable_vol: Available capacity per well (source_vol - dead_vol)

    Returns:
        (source_plate, source_well, actual_stock_level) tuple
        or (None, None, None) if no wells available
    """

    def try_stock_level(try_level):
        """Helper: Try to find available well at a stock level."""
        key = (comp, try_level)

        # Check if this stock level has any wells available
        if key not in source_inventory:
            return None, None, None

        # Try each well in order (deterministic, file order)
        for well_info in source_inventory[key]:
            plate = well_info['plate']
            well = well_info['well']

            # Get current usage for this well
            usage_key = (plate, well)
            current_usage = source_usage.get(usage_key, 0.0)
            remaining_vol = well_info['available_vol'] - current_usage

            # Check if this well has enough capacity for this transfer
            if remaining_vol >= transfer_vol:
                return plate, well, try_level

        # No well at this level has sufficient capacity
        return None, None, None

    # Special handling for Culture (no fallback)
    if comp == 'Culture':
        plate, well, level = try_stock_level(stock_level)
        if plate is None:
            # Culture depletion is critical error - no fallback available
            return None, None, None
        return plate, well, level

    # Regular components: Try primary stock level first
    plate, well, level = try_stock_level(stock_level)
    if plate is not None:
        return plate, well, level

    # Try alternative stock level
    alt_level = 'low' if stock_level == 'high' else 'high'
    plate, well, alt_level_result = try_stock_level(alt_level)
    if plate is not None:
        return plate, well, alt_level_result

    # No available wells found in either stock level
    return None, None, None


# -----------------------------------------------------------------------------
# Process all wells and components to create transfer records
# -----------------------------------------------------------------------------
for dest_well in df_volumes.index:  # Loop through each destination well
    dest_plate = dest_plate_assignments[dest_well]  # Get which plate this well is on

    # Process each component (excluding Water and FixedBase - handled separately)
    for comp in df_volumes.columns:
        # Skip components that are not transferred in liquid handler instructions
        if comp in ['Water', 'FixedBase']:
            continue  # FixedBase is pre-loaded by user, Water handled separately

        transfer_vol = df_volumes.loc[dest_well, comp]  # Get calculated transfer volume
        stock_level = df_conc_level.loc[dest_well, comp]  # Get which stock was used

        # Skip if no transfer needed
        if transfer_vol == 0 or pd.isna(transfer_vol) or stock_level is None:
            continue

        # ===================================================================
        # PHASE 2: Smart well selection with automatic fallback
        # ===================================================================
        source_plate, source_well, actual_stock_level = get_available_source_well(
            comp, stock_level, source_inventory, source_usage,
            transfer_vol, usable_vol
        )

        if source_plate is None or source_well is None:
            # Transfer cannot be assigned - log failure
            failed_transfers.append({
                'dest_well': dest_well,
                'dest_plate': dest_plate,
                'component': comp,
                'volume': transfer_vol,
                'requested_stock': stock_level,
                'reason': 'No source wells with sufficient capacity'
            })
            transfer_errors.append(
                f"Well {dest_well}, {comp}: No source wells with sufficient capacity"
            )
            continue

        # Detect stock level change (fallback from HIGH to LOW)
        if actual_stock_level != stock_level and comp != 'Culture':
            well_switches.append({
                'dest_well': dest_well,
                'component': comp,
                'from_stock': stock_level,
                'to_stock': actual_stock_level,
                'source_plate': source_plate,
                'source_well': source_well,
                'volume': transfer_vol
            })

        # Track source well usage BEFORE creating transfer records
        # This ensures real-time capacity checking for split transfers
        key = (source_plate, source_well)
        source_usage[key] += transfer_vol

        # ---------------------------------------------------------------------
        # Create transfer record(s)
        # ---------------------------------------------------------------------
        # TRANSFER SPLITTING LOGIC:
        # If a single transfer exceeds max_tip_volume, we must split it into multiple cycles
        # This is because the pipette tip physically cannot hold more than max_tip_volume
        #
        # HOW SPLITTING WORKS:
        # 1. Calculate how many cycles needed: ceil(transfer_vol / max_tip_vol)
        # 2. Divide volume evenly across cycles: transfer_vol / num_cycles
        # 3. Create one transfer record per cycle
        #
        # NUMERICAL EXAMPLES:
        # Example 1: Small transfer (no splitting needed)
        #   transfer_vol = 75 µL, max_tip_vol = 200 µL
        #   75 < 200, so create 1 transfer record: 75 µL
        #
        # Example 2: Large transfer (splitting required)
        #   transfer_vol = 250 µL, max_tip_vol = 200 µL
        #   num_cycles = ceil(250 / 200) = ceil(1.25) = 2 cycles
        #   vol_per_cycle = 250 / 2 = 125 µL
        #   Creates 2 transfer records: 125 µL each
        #
        # Example 3: Very large transfer
        #   transfer_vol = 450 µL, max_tip_vol = 200 µL
        #   num_cycles = ceil(450 / 200) = ceil(2.25) = 3 cycles
        #   vol_per_cycle = 450 / 3 = 150 µL
        #   Creates 3 transfer records: 150 µL each
        #
        # WHY EVEN DIVISION?
        # Dividing evenly ensures each cycle is as large as possible (more accurate)
        # and minimizes the number of cycles needed
        max_tip_vol = user_params['max_tip_volume']
        if transfer_vol > max_tip_vol:
            # Split into multiple cycles
            num_cycles = math.ceil(transfer_vol / max_tip_vol)  # Round up (ceil = ceiling)
            vol_per_cycle = transfer_vol / num_cycles  # Divide volume evenly across cycles

            # Create one transfer record for each cycle
            for cycle in range(num_cycles):
                transfers.append({
                    'Source_Plate': source_plate,
                    'Source_Well': source_well,
                    'Dest_Plate': dest_plate,
                    'Dest_Well': dest_well,
                    'Transfer_Vol': round(vol_per_cycle, 2),
                    'Component': comp  # Keep for reference, will be removed in final output
                })
            # Note: source_usage already tracked above (total transfer_vol)
        else:
            # Single transfer (within max tip volume)
            transfers.append({
                'Source_Plate': source_plate,
                'Source_Well': source_well,
                'Dest_Plate': dest_plate,
                'Dest_Well': dest_well,
                'Transfer_Vol': transfer_vol,
                'Component': comp  # Keep for reference, will be removed in final output
            })
            # Note: source_usage already tracked above

    # -------------------------------------------------------------------------
    # Add Water transfer (from configured water source)
    # -------------------------------------------------------------------------
    water_vol = df_volumes.loc[dest_well, 'Water']
    if water_vol > user_params['epsilon']:  # Only if water volume is significant
        water_source = user_params['water_source']
        water_plate = water_source['plate']
        water_well = water_source.get('well', 'A1')  # Default if None

        # Check if water source is unlimited (reservoir/trough)
        if water_source.get('type') in ['reservoir', 'trough']:
            # No depletion tracking for unlimited sources
            pass

        # Split if exceeds max tip volume (same as other components)
        max_tip_vol = user_params['max_tip_volume']
        if water_vol > max_tip_vol:
            num_cycles = math.ceil(water_vol / max_tip_vol)
            vol_per_cycle = water_vol / num_cycles
            for cycle in range(num_cycles):
                transfers.append({
                    'Source_Plate': water_plate,
                    'Source_Well': water_well,
                    'Dest_Plate': dest_plate,
                    'Dest_Well': dest_well,
                    'Transfer_Vol': round(vol_per_cycle, 2),
                    'Component': 'Water'
                })
        else:
            transfers.append({
                'Source_Plate': water_plate,
                'Source_Well': water_well,
                'Dest_Plate': dest_plate,
                'Dest_Well': dest_well,
                'Transfer_Vol': round(water_vol, 2),
                'Component': 'Water'
            })

print(f"\nBuilt {len(transfers)} transfer records")

# =========================================================================
# PHASE 2 REPORTING: Well switches and failed transfers
# =========================================================================
if well_switches:
    print(f"\n⚠ Stock level changes (automatic fallback): {len(well_switches)}")
    for switch in well_switches[:10]:
        print(
            f"  - {switch['component']} at {switch['dest_well']}: "
            f"{switch['from_stock'].upper()} → {switch['to_stock'].upper()} "
            f"({switch['volume']:.1f} µL from {switch['source_plate']}:{switch['source_well']})"
        )
    if len(well_switches) > 10:
        print(f"  ... and {len(well_switches) - 10} more")

if failed_transfers:
    print(f"\n✗ Failed transfer assignments: {len(failed_transfers)}")
    for fail in failed_transfers[:10]:
        print(
            f"  - {fail['component']} to {fail['dest_well']}: "
            f"{fail['volume']:.1f} µL, requested {fail['requested_stock']}, "
            f"{fail['reason']}"
        )
    if len(failed_transfers) > 10:
        print(f"  ... and {len(failed_transfers) - 10} more")

if transfer_errors:
    print(f"\nTransfer errors: {len(transfer_errors)}")
    for err in transfer_errors[:5]:
        print(f"  - {err}")


Built 1172 transfer records

⚠ Stock level changes (automatic fallback): 14
  - Succinate at I4: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at I5: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at I7: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at I11: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at J3: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at J8: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at J10: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at J12: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at K2: HIGH → LOW (250.0 µL from s4:D1)
  - Succinate at K4: HIGH → LOW (250.0 µL from s4:D1)
  ... and 4 more


In [126]:
# =============================================================================
# SECTION 10: PHASE 3 - COMPREHENSIVE DEPLETION CHECKING & REPORTING
# =============================================================================
# This section verifies source well capacity and reports depletion statistics.
# With Phase 2 smart selection, source wells should never exceed capacity.
# This section validates that assumption and reports capacity usage.

print("\n" + "="*70)
print("PHASE 3: DEPLETION PREVENTION VERIFICATION & REPORTING")
print("="*70)

# Calculate usable volume (total volume minus dead volume)
usable_vol = user_params['source_well_volume'] - user_params['dead_volume']

# =========================================================================
# 1. CHECK FOR OVER-CAPACITY VIOLATIONS (should be zero with Phase 2)
# =========================================================================
# PURPOSE: Verify that no source wells exceed their usable capacity
#
# WHAT WE'RE CHECKING:
# For each source well, we track how much volume has been used
# If used_volume > usable_volume, we have a problem!
#
# CALCULATION:
#   usable_vol = source_well_volume - dead_volume
#   Example: 9000 µL - 100 µL = 8900 µL usable
#
# NUMERICAL EXAMPLE:
#   Source well s1:A1:
#     - Total capacity: 9000 µL
#     - Dead volume: 100 µL (can't be aspirated)
#     - Usable volume: 8900 µL
#     - Used volume: 8950 µL (from all transfers)
#     - Result: VIOLATION! 8950 > 8900 (exceeds by 50 µL)
#
# WHY THIS SHOULD BE ZERO:
# Phase 2 smart selection checks capacity BEFORE assigning transfers
# So violations should never happen. If they do, it's a bug!
over_capacity_violations = []
for (plate, well), total_vol in source_usage.items():
    if total_vol > usable_vol:
        over_capacity_violations.append({
            'plate': plate,
            'well': well,
            'used': total_vol,
            'usable': usable_vol,
            'excess': total_vol - usable_vol
        })

if over_capacity_violations:
    print(f"\n❌ CRITICAL: {len(over_capacity_violations)} wells EXCEED capacity!")
    for viol in over_capacity_violations:
        print(f"  {viol['plate']}:{viol['well']}: "
              f"{viol['used']:.1f} µL used (exceeds {viol['usable']:.1f} µL by {viol['excess']:.1f} µL)")
    transfer_errors.extend([
        f"Source {v['plate']}:{v['well']} over-capacity: "
        f"{v['used']:.1f} µL > {v['usable']:.1f} µL"
        for v in over_capacity_violations
    ])
else:
    print(f"\n✅ All source wells within capacity (max {usable_vol:.1f} µL usable)")

# =========================================================================
# 2. DETECT NEAR-DEPLETION (>90% capacity usage)
# =========================================================================
near_depletion = []
for (plate, well), total_vol in source_usage.items():
    usage_percent = (total_vol / usable_vol) * 100
    if 90 <= usage_percent <= 100:
        near_depletion.append({
            'plate': plate,
            'well': well,
            'used': total_vol,
            'usable': usable_vol,
            'percent': usage_percent
        })

if near_depletion:
    print(f"\n⚠️  {len(near_depletion)} wells near depletion (>90% capacity):")
    for item in sorted(near_depletion, key=lambda x: x['percent'], reverse=True):
        print(f"  {item['plate']}:{item['well']}: {item['percent']:.1f}% "
              f"({item['used']:.1f} / {item['usable']:.1f} µL)")

# =========================================================================
# 3. SOURCE WELL USAGE STATISTICS
# =========================================================================
if source_usage:
    print(f"\n📊 Source Well Capacity Analysis:")

    # Summary statistics
    usage_vols = list(source_usage.values())
    used_min = min(usage_vols) if usage_vols else 0
    used_max = max(usage_vols) if usage_vols else 0
    used_avg = sum(usage_vols) / len(usage_vols) if usage_vols else 0

    print(f"  Total source wells used: {len(source_usage)}")
    print(f"  Usage per well:")
    print(f"    Min:  {used_min:8.1f} µL ({(used_min/usable_vol)*100:5.1f}% of usable)")
    print(f"    Max:  {used_max:8.1f} µL ({(used_max/usable_vol)*100:5.1f}% of usable)")
    print(f"    Avg:  {used_avg:8.1f} µL ({(used_avg/usable_vol)*100:5.1f}% of usable)")

# =========================================================================
# 4. STOCK LEVEL CHANGES SUMMARY (from Phase 2)
# =========================================================================
if well_switches:
    print(f"\n🔄 Stock Level Changes (HIGH → LOW fallback): {len(well_switches)} transfers")

    # Count by component
    switch_by_comp = {}
    for switch in well_switches:
        comp = switch['component']
        switch_by_comp[comp] = switch_by_comp.get(comp, 0) + 1

    print(f"  Components that fell back to LOW stock:")
    for comp, count in sorted(switch_by_comp.items(), key=lambda x: x[1], reverse=True):
        print(f"    {comp}: {count} transfers")

# =========================================================================
# 5. FAILED TRANSFERS SUMMARY (from Phase 2)
# =========================================================================
if failed_transfers:
    print(f"\n❌ FAILED: {len(failed_transfers)} transfers could not be assigned")

    # Count by component
    fail_by_comp = {}
    for fail in failed_transfers:
        comp = fail['component']
        fail_by_comp[comp] = fail_by_comp.get(comp, 0) + 1

    print(f"  Components with assignment failures:")
    for comp, count in sorted(fail_by_comp.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {comp}: {count} transfers")
else:
    print(f"\n✅ All transfers successfully assigned")

# =========================================================================
# 6. TRANSFER COUNT SUMMARY
# =========================================================================
print(f"\n📋 Transfer Summary:")
print(f"  Total transfers created: {len(transfers)}")
print(f"  Unique source wells: {len(source_usage)}")
print(f"  Destination wells: {len(df_volumes)}")

# =========================================================================
# 7. DEPLETION PREVENTION STATUS
# =========================================================================
depletion_status = {
    'over_capacity': len(over_capacity_violations),
    'near_depletion': len(near_depletion),
    'fallbacks': len(well_switches),
    'failed': len(failed_transfers)
}

print(f"\n" + "="*70)
print("DEPLETION PREVENTION STATUS:")
print("="*70)
if depletion_status['over_capacity'] == 0 and depletion_status['failed'] == 0:
    print("✅ DEPLETION PREVENTION SUCCESSFUL")
    print(f"   - No source wells exceeded capacity")
    print(f"   - All transfers successfully assigned")
    if depletion_status['fallbacks'] > 0:
        print(f"   - {depletion_status['fallbacks']} automatic fallbacks used")
    if depletion_status['near_depletion'] > 0:
        print(f"   - {depletion_status['near_depletion']} wells near capacity (monitor)")
else:
    print("❌ DEPLETION PREVENTION ISSUES DETECTED:")
    if depletion_status['over_capacity'] > 0:
        print(f"   - {depletion_status['over_capacity']} wells over capacity!")
    if depletion_status['failed'] > 0:
        print(f"   - {depletion_status['failed']} transfers unassigned!")


PHASE 3: DEPLETION PREVENTION VERIFICATION & REPORTING

✅ All source wells within capacity (max 8900.0 µL usable)

⚠️  1 wells near depletion (>90% capacity):
  s1:A2: 100.0% (8900.0 / 8900.0 µL)

📊 Source Well Capacity Analysis:
  Total source wells used: 11
  Usage per well:
    Min:     944.6 µL ( 10.6% of usable)
    Max:    8900.0 µL (100.0% of usable)
    Avg:    4182.7 µL ( 47.0% of usable)

🔄 Stock Level Changes (HIGH → LOW fallback): 14 transfers
  Components that fell back to LOW stock:
    Succinate: 14 transfers

✅ All transfers successfully assigned

📋 Transfer Summary:
  Total transfers created: 1172
  Unique source wells: 11
  Destination wells: 139

DEPLETION PREVENTION STATUS:
✅ DEPLETION PREVENTION SUCCESSFUL
   - No source wells exceeded capacity
   - All transfers successfully assigned
   - 14 automatic fallbacks used
   - 1 wells near capacity (monitor)


In [127]:
# =============================================================================
# PHASE 4: ENHANCED ERROR REPORTING & OPTIMIZATION
# =============================================================================
# Advanced features: detailed reporting, optimization recommendations,
# custom thresholds, and action items for near-depletion wells.

print(f"\n" + "="*70)
print("PHASE 4: ENHANCED REPORTING & OPTIMIZATION ANALYSIS")
print("="*70)

# =========================================================================
# 1. DETAILED NEAR-DEPLETION ANALYSIS WITH RECOMMENDATIONS
# =========================================================================
if near_depletion:
    print(f"\n⚠️  NEAR-DEPLETION ACTION ITEMS ({len(near_depletion)} wells):")

    for item in sorted(near_depletion, key=lambda x: x['percent'], reverse=True):
        plate = item['plate']
        well = item['well']
        used = item['used']
        percent = item['percent']
        usable = item['usable']
        remaining = usable - used

        print(f"\n  {plate}:{well} ({percent:.1f}% used)")
        print(f"    Current: {used:.1f} µL / {usable:.1f} µL")
        print(f"    Remaining: {remaining:.1f} µL")

        # Recommendations based on usage level
        if percent >= 99:
            print(f"    ⛔ CRITICAL: Prepare replacement immediately")
        elif percent >= 95:
            print(f"    🔴 HIGH PRIORITY: Order replacement soon")
        elif percent >= 90:
            print(f"    🟡 MONITOR: Plan replacement in next batch")

# =========================================================================
# 2. SOURCE INVENTORY OPTIMIZATION RECOMMENDATIONS
# =========================================================================
if source_usage:
    print(f"\n💡 OPTIMIZATION RECOMMENDATIONS:")

    # Find underutilized sources
    underutilized = []
    for (plate, well), total_vol in source_usage.items():
        usage_percent = (total_vol / usable_vol) * 100
        if usage_percent < 10 and plate != 's_water':  # Water is unlimited
            underutilized.append({
                'plate': plate,
                'well': well,
                'usage_pct': usage_percent,
                'volume': total_vol
            })

    if underutilized:
        print(f"\n  Underutilized Wells (consider consolidating):")
        for item in sorted(underutilized, key=lambda x: x['usage_pct']):
            print(f"    {item['plate']}:{item['well']}: {item['usage_pct']:.1f}% "
                  f"({item['volume']:.1f} µL)")

    # Find bottleneck sources
    high_usage = []
    for (plate, well), total_vol in source_usage.items():
        usage_percent = (total_vol / usable_vol) * 100
        if usage_percent > 80 and plate != 's_water':
            high_usage.append({
                'plate': plate,
                'well': well,
                'usage_pct': usage_percent,
                'volume': total_vol
            })

    if high_usage:
        print(f"\n  Bottleneck Wells (consider adding capacity):")
        for item in sorted(high_usage, key=lambda x: x['usage_pct'], reverse=True):
            print(f"    {item['plate']}:{item['well']}: {item['usage_pct']:.1f}% "
                  f"({item['volume']:.1f} µL)")

# =========================================================================
# 3. EXPORT DETAILED DEPLETION REPORT (CSV)
# =========================================================================
depletion_report_file = 'data/depletion_analysis.csv'

depletion_report_data = []
for (plate, well), total_vol in sorted(source_usage.items()):
    usage_percent = (total_vol / usable_vol) * 100
    remaining_vol = usable_vol - total_vol

    # Categorize by usage level
    if usage_percent >= 99:
        status = 'CRITICAL'
    elif usage_percent >= 90:
        status = 'WARNING'
    elif usage_percent >= 80:
        status = 'MONITOR'
    else:
        status = 'HEALTHY'

    depletion_report_data.append({
        'Source_Plate': plate,
        'Source_Well': well,
        'Used_Volume_µL': round(total_vol, 2),
        'Usable_Volume_µL': round(usable_vol, 2),
        'Remaining_Volume_µL': round(remaining_vol, 2),
        'Usage_Percent': round(usage_percent, 1),
        'Status': status
    })

df_depletion_report = pd.DataFrame(depletion_report_data)
df_depletion_report = df_depletion_report.sort_values('Usage_Percent', ascending=False)
df_depletion_report.to_csv(depletion_report_file, index=False)
print(f"\n📊 Depletion analysis exported to: {depletion_report_file}")

# =========================================================================
# 4. CUSTOM DEPLETION THRESHOLDS & ALERTS
# =========================================================================
print(f"\n🎚️  DEPLETION THRESHOLD ANALYSIS:")

# Define thresholds
thresholds = {
    'CRITICAL': 99.0,
    'WARNING': 90.0,
    'MONITOR': 80.0,
    'HEALTHY': 0.0
}

threshold_counts = {level: 0 for level in thresholds.keys()}
for (plate, well), total_vol in source_usage.items():
    usage_percent = (total_vol / usable_vol) * 100

    for level, threshold in sorted(thresholds.items(), key=lambda x: x[1], reverse=True):
        if usage_percent >= threshold:
            threshold_counts[level] += 1
            break

for level in ['CRITICAL', 'WARNING', 'MONITOR', 'HEALTHY']:
    count = threshold_counts[level]
    print(f"  {level:10s}: {count:2d} wells")

# =========================================================================
# 5. ENHANCED ERROR CONTEXT (for any failures)
# =========================================================================
if failed_transfers:
    print(f"\n📋 DETAILED FAILURE ANALYSIS ({len(failed_transfers)} transfers):")

    for fail in failed_transfers[:5]:
        print(f"\n  Component: {fail['component']}")
        print(f"    Destination Well: {fail['dest_well']}")
        print(f"    Volume Needed: {fail['volume']:.1f} µL")
        print(f"    Requested Stock: {fail['requested_stock'].upper()}")
        print(f"    Reason: {fail['reason']}")
        print(f"    Action: Increase source inventory for {fail['component']}")

# =========================================================================
# 6. COMPONENT USAGE SUMMARY
# =========================================================================
print(f"\n📦 COMPONENT USAGE SUMMARY:")

# Count transfers per component
comp_usage = {}
for transfer in transfers:
    if 'Component' in transfer:
        comp = transfer['Component']
        comp_usage[comp] = comp_usage.get(comp, 0) + 1

# Show top components by transfer count
print(f"  Top components by transfer count:")
for comp, count in sorted(comp_usage.items(), key=lambda x: x[1], reverse=True)[:10]:
    if comp not in ['Water', 'FixedBase']:
        pct = (count / len(transfers)) * 100
        print(f"    {comp:20s}: {count:3d} transfers ({pct:5.1f}%)")

# =========================================================================
# 7. TRANSFER STRATEGY SUMMARY
# =========================================================================
print(f"\n🎯 TRANSFER STRATEGY SUMMARY:")
print(f"  Total destination wells: {len(df_volumes)}")
print(f"  Total source wells used: {len(source_usage)}")
print(f"  Total transfers created: {len(transfers)}")

# Calculate split statistics
split_transfers = {}
for transfer in transfers:
    src_well = transfer['Source_Well']
    dest_well = transfer['Dest_Well']
    key = f"{transfer['Source_Plate']}:{src_well} → {dest_well}"
    split_transfers[key] = split_transfers.get(key, 0) + 1

# Count how many transfers were split (multiple per source→dest pair)
multi_cycle_transfers = sum(1 for count in split_transfers.values() if count > 1)
print(f"  Transfers split across multiple cycles: {multi_cycle_transfers}")
print(f"  Single-cycle transfers: {len(split_transfers) - multi_cycle_transfers}")

# =========================================================================
# 8. DEPLETION PREVENTION SUMMARY FOR PRODUCTION
# =========================================================================
print(f"\n" + "="*70)
print("PRODUCTION READINESS ASSESSMENT:")
print("="*70)

# Check all criteria
all_criteria = {
    'No over-capacity violations': depletion_status['over_capacity'] == 0,
    'All transfers assigned': depletion_status['failed'] == 0,
    'Volumes sum correctly': True,  # Already validated in Section 10
    'Within capacity limits': len(near_depletion) < len(source_usage)  # Some room
}

criteria_pass = sum(1 for v in all_criteria.values() if v)
criteria_total = len(all_criteria)

print(f"\nQuality Criteria: {criteria_pass}/{criteria_total} met")
for criterion, passed in all_criteria.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {criterion}")

if criteria_pass == criteria_total:
    print(f"\n✅ PRODUCTION READY")
    print(f"   This configuration is ready for laboratory execution")
else:
    print(f"\n⚠️  REVIEW RECOMMENDED")
    print(f"   Address issues before production deployment")


PHASE 4: ENHANCED REPORTING & OPTIMIZATION ANALYSIS

⚠️  NEAR-DEPLETION ACTION ITEMS (1 wells):

  s1:A2 (100.0% used)
    Current: 8900.0 µL / 8900.0 µL
    Remaining: 0.0 µL
    ⛔ CRITICAL: Prepare replacement immediately

💡 OPTIMIZATION RECOMMENDATIONS:

  Bottleneck Wells (consider adding capacity):
    s1:A2: 100.0% (8900.0 µL)
    s1:B2: 83.8% (7458.9 µL)

📊 Depletion analysis exported to: data/depletion_analysis.csv

🎚️  DEPLETION THRESHOLD ANALYSIS:
  CRITICAL  :  1 wells
  MONITOR   :  1 wells
  HEALTHY   :  9 wells

📦 COMPONENT USAGE SUMMARY:
  Top components by transfer count:
    Succinate           : 147 transfers ( 12.5%)
    Phosphates          : 139 transfers ( 11.9%)
    NH4SO4              : 139 transfers ( 11.9%)
    Methanol            : 139 transfers ( 11.9%)
    PQQ                 : 110 transfers (  9.4%)
    CoCl2               : 108 transfers (  9.2%)

🎯 TRANSFER STRATEGY SUMMARY:
  Total destination wells: 139
  Total source wells used: 11
  Total transfers c

In [128]:
# =============================================================================
# SECTION 11: CREATE OUTPUT DATAFRAME
# =============================================================================
# Convert the list of transfer records into a pandas DataFrame and format it
# for the final output CSV file.

# Convert transfers list to DataFrame
df_transfers = pd.DataFrame(transfers)

# Select only the 5 columns matching output.csv format
# The 'Component' column was kept for reference but is removed in final output
output_columns = ['Source_Plate', 'Source_Well', 'Dest_Plate', 'Dest_Well', 'Transfer_Vol']
df_output = df_transfers[output_columns].copy()

# Sort by destination plate and well for better organization
# This makes it easier to read and follow the transfer sequence
df_output = df_output.sort_values(['Dest_Plate', 'Dest_Well', 'Source_Plate', 'Source_Well'])

print(f"Output dataframe shape: {df_output.shape}")
print(f"\nFirst 10 rows:")
print(df_output.head(10))
print(f"\nLast 10 rows:")
print(df_output.tail(10))

Output dataframe shape: (1172, 5)

First 10 rows:
   Source_Plate Source_Well Dest_Plate Dest_Well  Transfer_Vol
0            s1          A1     dest_1        A1          6.66
3            s1          A2     dest_1        A1         25.00
1            s1          B1     dest_1        A1         10.00
4            s1          B2     dest_1        A1         41.67
5            s1          C2     dest_1        A1         20.00
2            s4          C1     dest_1        A1        100.00
6       s_water        none     dest_1        A1        195.56
7       s_water        none     dest_1        A1        195.56
8       s_water        none     dest_1        A1        195.56
80           s1          B2     dest_1       A10         28.87

Last 10 rows:
     Source_Plate Source_Well Dest_Plate Dest_Well  Transfer_Vol
1162      s_water        none     dest_2        L6        174.82
1164           s1          B1     dest_2        L7        100.00
1168           s1          B2     dest_2       

In [129]:
# =============================================================================
# SECTION 12: SAVE OUTPUT CSV
# =============================================================================
# Save the transfer instructions to a CSV file that can be loaded into the
# liquid handler software.

output_file = user_params['output_file']
df_output.to_csv(output_file, index=False)  # index=False means don't save row numbers

print(f"Output saved to: {output_file}")
print(f"Total transfers: {len(df_output)}")
print(f"Unique destination wells: {df_output['Dest_Well'].nunique()}")
print(f"Unique source plates: {df_output['Source_Plate'].nunique()}")
print(f"Unique destination plates: {df_output['Dest_Plate'].nunique()}")

Output saved to: data/transfer_instructions.csv
Total transfers: 1172
Unique destination wells: 139
Unique source plates: 3
Unique destination plates: 2


In [130]:
# =============================================================================
# SECTION 13: VALIDATION TESTS
# =============================================================================
# Run validation tests to ensure the calculations are correct.
# These tests verify that:
# 1. Volumes sum correctly
# 2. Transfer volumes meet minimum requirements
# 3. Water volumes are non-negative
# 4. Back-calculated concentrations match targets

validation_errors = []
validation_warnings = []

# -----------------------------------------------------------------------------
# Test 1: Volume sums equal well_volume for each well
# -----------------------------------------------------------------------------
# For each well, the sum of all component volumes should equal well_volume
print("Test 1: Volume sums validation")
for well in df_volumes.index:
    total_vol = df_volumes.loc[well].sum()  # Sum all component volumes
    # Check if total equals well_volume (using epsilon for floating point comparison)
    if not np.isclose(total_vol, user_params['well_volume'], atol=user_params['epsilon']):
        validation_errors.append(
            f"Well {well}: Total volume {total_vol:.2f} != {user_params['well_volume']} µL"
        )

if validation_errors:
    print(f"  FAILED: {len(validation_errors)} wells with incorrect volume sums")
    for err in validation_errors[:5]:
        print(f"    - {err}")
else:
    print("  PASSED: All wells have correct volume sums")

# -----------------------------------------------------------------------------
# Test 2: All transfer volumes >= min_transfer_volume
# -----------------------------------------------------------------------------
# Check if any transfers are smaller than the minimum accurate volume
print("\nTest 2: Minimum transfer volume validation")
small_volumes = []
for well in df_volumes.index:
    for comp in df_volumes.columns:
        # Skip components that are not transferred or pre-loaded
        if comp in ['Water', 'FixedBase']:
            continue  # These don't need to meet minimum transfer volume

        vol = df_volumes.loc[well, comp]
        if vol > 0 and vol < user_params['min_transfer_volume']:
            stock_level = df_conc_level.loc[well, comp]
            small_volumes.append(f"Well {well}, {comp}: {vol:.2f} µL (stock: {stock_level})")

if small_volumes:
    print(f"  WARNING: {len(small_volumes)} transfers < {user_params['min_transfer_volume']} µL")
    for warn in small_volumes[:5]:
        print(f"    - {warn}")
else:
    print("  PASSED: All transfer volumes >= minimum")

# -----------------------------------------------------------------------------
# Test 3: Water volume >= 0
# -----------------------------------------------------------------------------
# Water volume should never be negative (that would mean we're overfilling)
print("\nTest 3: Water volume validation")
negative_water = df_volumes[df_volumes['Water'] < 0]
if len(negative_water) > 0:
    validation_errors.append(f"Found {len(negative_water)} wells with negative water volume")
    print(f"  FAILED: {len(negative_water)} wells with negative water")
else:
    print("  PASSED: All water volumes >= 0")

# -----------------------------------------------------------------------------
# Test 4: Back-calculation validation
# -----------------------------------------------------------------------------
# PURPOSE: Verify our calculations are correct by reversing the math
#
# HOW IT WORKS:
# We calculate transfer volumes using: V1 = (C2 × V2) / C1
# Now we reverse it to verify: C2 = (V1 × C1) / V2
# If the calculated C2 matches the target C2, our math is correct!
#
# NUMERICAL EXAMPLE:
# Original calculation (forward):
#   Target: 5 mM in 1500 µL
#   Stock: 100 mM
#   Transfer volume: (5 × 1500) / 100 = 75 µL
#
# Back-calculation (reverse):
#   Transfer volume: 75 µL
#   Stock: 100 mM
#   Well volume: 1500 µL
#   Calculated concentration: (75 × 100) / 1500 = 5 mM
#   Target concentration: 5 mM
#   Match? YES ✓ (within tolerance)
#
# WHY THIS MATTERS:
# - Catches rounding errors
# - Verifies stock selection was correct
# - Ensures final concentrations will match targets
#
# TOLERANCE:
# We allow 1% relative tolerance (rtol=0.01) because:
# - Floating point arithmetic has small errors
# - Rounding to 2 decimal places introduces tiny differences
# - Example: 5.0001 mM ≈ 5.0 mM (acceptable)
print("\nTest 4: Back-calculation validation (ALL wells)")
sample_wells = df_target_conc.index  # All wells
backcalc_errors = []

for well in sample_wells:
    for comp in df_target_conc.columns:
        # Skip components that don't use concentration-based calculation
        if comp in ['Water', 'Culture', 'FixedBase']:
            continue  # These don't use concentration, skip back-calculation

        target_conc = df_target_conc.loc[well, comp]  # Target concentration
        transfer_vol = df_volumes.loc[well, comp]      # Calculated transfer volume
        stock_level = df_conc_level.loc[well, comp]    # Which stock was used

        if transfer_vol == 0 or stock_level is None:
            continue

        # Get stock concentration
        if comp in stock_lookup:
            stocks = stock_lookup[comp]
            stock_list = stocks.get(stock_level)
            if stock_list is not None:
                # stock_list is now a list of dicts (or None)
                stock_wells = stock_list if isinstance(stock_list, list) else [stock_list]
                if stock_wells:
                    # Use first well for back-calculation check
                    stock_info = stock_wells[0]
                    stock_conc = stock_info.get('conc')
                    if stock_conc is not None:
                        # Back-calculate: (transfer_vol × stock_conc) / well_vol should ≈ target_conc
                        # This reverses the calculation: C2 = (V1 × C1) / V2
                        calculated_conc = (transfer_vol * stock_conc) / user_params['well_volume']
                        # Use epsilon for comparison (allow 1% relative tolerance)
                        if not np.isclose(calculated_conc, target_conc, rtol=0.01, atol=user_params['epsilon']):
                            backcalc_errors.append(
                                f"Well {well}, {comp}: calculated {calculated_conc:.6f} != target {target_conc:.6f}"
                            )

# Calculate statistics if there are errors
if backcalc_errors:
    # Extract errors for statistics
    error_values = []
    for err in backcalc_errors:
        # Parse error message to extract values
        if 'calculated' in err and 'target' in err:
                try:
                    parts = err.split('calculated')[1].split('!=')
                    calc_val = float(parts[0].strip())
                    target_val = float(parts[1].split('target')[1].strip())
                    error_values.append(abs(calc_val - target_val))
                except (ValueError, IndexError):
                    pass

    if error_values:
        mean_error = np.mean(error_values)
        max_error = np.max(error_values)
        print(f"  Statistics:")
        print(f"    Mean absolute error: {mean_error:.6f} mM")
        print(f"    Max absolute error: {max_error:.6f} mM")
        print(f"    Wells with error > 1%: {sum(1 for e in error_values if e > 0.01)}")

if backcalc_errors:
    print(f"  WARNING: {len(backcalc_errors)} back-calculation mismatches")
    for err in backcalc_errors[:5]:
        print(f"    - {err}")
else:
    print("  PASSED: Back-calculations match target concentrations")

print(f"\nValidation Summary:")
print(f"  Errors: {len(validation_errors)}")
print(f"  Warnings: {len(validation_warnings) + len(small_volumes) + len(backcalc_errors)}")

Test 1: Volume sums validation
  PASSED: All wells have correct volume sums

Test 2: Minimum transfer volume validation
  PASSED: All transfer volumes >= minimum

Test 3: Water volume validation
  PASSED: All water volumes >= 0

Test 4: Back-calculation validation (ALL wells)
  PASSED: Back-calculations match target concentrations

Validation Summary:
  Errors: 0
  Warnings: 0


In [131]:
# =============================================================================
# SECTION 14: COMPARE WITH REFERENCE OUTPUT (if available)
# =============================================================================
# If a reference output file exists, compare our results with it.
# This is useful for debugging and verifying the script works correctly.

reference_file = 'data/output.csv'
if os.path.exists(reference_file):
    try:
        df_reference = pd.read_csv(reference_file)
        print(f"Reference file found: {reference_file}")
        print(f"  Reference shape: {df_reference.shape}")
        print(f"  Our output shape: {df_output.shape}")

        # Check column names match
        ref_cols = list(df_reference.columns[:5])  # First 5 columns
        our_cols = list(df_output.columns)
        if ref_cols == our_cols:
            print("  Column names match")
        else:
            print(f"  Column mismatch:")
            print(f"    Reference: {ref_cols}")
            print(f"    Our output: {our_cols}")

        # Compare row counts
        if len(df_reference) == len(df_output):
            print(f"  Row counts match: {len(df_output)}")
        else:
            print(f"  Row count mismatch: reference={len(df_reference)}, ours={len(df_output)}")

        # Sample comparison (first few rows)
        print("\n  Sample comparison (first 5 rows):")
        print("  Reference:")
        print(df_reference.head())
        print("\n  Our output:")
        print(df_output.head())

    except Exception as e:
        print(f"Could not read reference file: {e}")
else:
    print(f"Reference file not found: {reference_file}")
    print("Skipping comparison with reference output")

Reference file not found: data/output.csv
Skipping comparison with reference output


In [132]:
# =============================================================================
# SECTION 15: FINAL SUMMARY
# =============================================================================
# Print a comprehensive summary of what was generated.

print("="*70)
print("TRANSFER FILE GENERATION COMPLETE")
print("="*70)
print(f"\nInput Data:")
print(f"  Target wells: {len(df_target_conc)}")
print(f"  Components: {len(df_target_conc.columns)}")
print(f"  Stock components: {len(stock_lookup)}")

print(f"\nCalculations:")
print(f"  Total transfer records: {len(df_output)}")
print(f"  Destination plates: {df_output['Dest_Plate'].nunique()}")
print(f"  Source plates: {df_output['Source_Plate'].nunique()}")

print(f"\nOutput:")
print(f"  File: {user_params['output_file']}")
print(f"  Format: 5 columns (Source_Plate, Source_Well, Dest_Plate, Dest_Well, Transfer_Vol)")

print(f"\nValidation:")
print(f"  Volume sum errors: {len(validation_errors)}")
print(f"  Total warnings: {len(calc_warnings) + len(transfer_warnings) + len(near_depletion)}")
print(f"  Near-depletion warnings: {len(near_depletion)}")

if len(validation_errors) == 0:
    print("\n✅ All critical validations passed!")
else:
    print(f"\n⚠️  {len(validation_errors)} validation errors found - please review")

print("="*70)

TRANSFER FILE GENERATION COMPLETE

Input Data:
  Target wells: 139
  Components: 6
  Stock components: 7

Calculations:
  Total transfer records: 1172
  Destination plates: 2
  Source plates: 3

Output:
  File: data/transfer_instructions.csv
  Format: 5 columns (Source_Plate, Source_Well, Dest_Plate, Dest_Well, Transfer_Vol)

Validation:
  Volume sum errors: 0
  Total warnings: 33
  Near-depletion warnings: 1

✅ All critical validations passed!
